# **Momento Evaluativo #2: Redes Neuronales Convolucionales**

## **Objetivo**:
Implementar una red neuronal convolucional y observar la diferencia entre un MLP y una CNN, además de realizar *transfer learning* en un problema de clasificación de imágenes.

## **Integrantes:**
-
-
-


## **Fecha de entrega:**
Domingo, 26 de abril

# **Red Neuronal Densamente Conectada (Fully Connected Neural Networks)**

In [ ]:
%matplotlib inline

import cv2
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras

In [ ]:
def plot_training_history(history):
    """
    Función para graficar cada métrica de entrenamiento y validación de un objeto history de TensorFlow en gráficas separadas.
    """
    # Recorre todas las claves en el diccionario history.history
    for metric in history.history.keys():
        # Verifica si la métrica es de validación o de entrenamiento
        if 'val_' in metric:
            continue  # Saltar métricas de validación para graficarlas junto con la métrica de entrenamiento

        # Define el nombre de la métrica de validación correspondiente (si existe)
        val_metric = f'val_{metric}'

        # Crear una nueva gráfica para cada métrica
        plt.figure()

        # Graficar la métrica de entrenamiento
        plt.plot(history.history[metric], label=f'Train {metric}')

        # Graficar la métrica de validación si está disponible
        if val_metric in history.history:
            plt.plot(history.history[val_metric], label=f'Validation {metric}')

        # Configuraciones de la gráfica
        plt.xlabel('Épocas')
        plt.ylabel(metric.capitalize())
        plt.title(f'Evolución de {metric}')
        plt.legend()
        plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorflow import keras

def plot_10_predictions(model, x_test, y_test, yname):
    """
    Función para mostrar 10 figuras con imágenes de prueba y sus respectivas predicciones.

    Parámetros:
        model: El modelo de TensorFlow/Keras entrenado.
        x_test: El conjunto de datos de prueba (imágenes).
        y_test: Las etiquetas reales correspondientes a las imágenes de prueba.
        yname: Lista o diccionario que mapea las etiquetas numéricas a nombres de clases.
    """
    for i in range(10):
        # Selecciona un índice aleatorio para la imagen de prueba
        n_test = np.random.randint(0, len(x_test))
        print(f"Imagen seleccionada: {n_test}")

        # Carga la imagen y la etiqueta correspondiente
        Im_prueba = x_test[n_test, ...]
        Etiq_prueba = y_test[n_test]

        # Predice la clase de la imagen
        clase_pred = keras.backend.argmax(model.predict(Im_prueba[None, ...])).numpy()
        Etiq_prueba = np.argmax(Etiq_prueba)


        # Configura la visualización de la imagen
        plt.figure(figsize=(3, 3))
        plt.imshow(np.reshape(Im_prueba, (32, 32,3)), cmap='gray')

        # Define el color del título según si la predicción es correcta o incorrecta
        color = 'blue' if clase_pred[0] == Etiq_prueba else 'red'
        plt.title(f"Predicha: {yname[clase_pred[0]]} (Real: {yname[Etiq_prueba]})", color=color)
        plt.axis('off')  # Opcional: oculta los ejes para una visualización más limpia
        plt.show()

# Ejemplo de uso de la función
# plot_10_predictions(model, x_test, y_test, yname)


## **Cargar conjunto de datos**

Primero, vamos a cargar el conjunto de datos. Para elementos prácticos, vamos a utilizar un conjunto de datos pre-cargados por Tensorflow. Vamos a utilizar el conjunto de datos CIFAR 10, el cual es una colección de imágenes ampliamente utilizada para el entrenamiento y prueba de algoritmos de visión por computadora.

Consiste en 60,000 imágenes en color de baja resolución (32x32 píxeles) divididas en 10 categorías: aviones, automóviles, pájaros, gatos, ciervos, perros, ranas, caballos, barcos y camiones. Cada categoría contiene 6,000 imágenes. Las imágenes están ya etiquetadas, lo que facilita su uso para tareas de clasificación.

Link base de datos: https://www.tensorflow.org/api_docs/python/tf/keras/datasets


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

print(f"Dimension de X train:{x_train.shape}")
print(f"Dimension de X test:{x_test.shape}")
print(f"Dimension de y train:{y_train.shape}")
print(f"Dimension de y test:{y_test.shape}")

yname = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

In [ ]:
# Graficar imagenes de muestra
fig, axes = plt.subplots(3,3)

axes = axes.ravel()

for ax in axes:

    n = np.random.randint(0,50000)
    ax.imshow(x_train[n,...])
    ax.set_title(f"Clase: {y_train[n]} - {yname[y_train[n][0]]}")
    ax.set_axis_off()

In [ ]:
# Escalar las imágenes en el rango [0,1]
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

print(f"Dimension de X train:{x_train.shape}")
print(f"Dimension de X test:{x_test.shape}")
print(f"Dimension de y train:{y_train.shape}")
print(f"Dimension de y test:{y_test.shape}")


## **Perceptrón MultiCapa**

Inicialmente, vamos a crear un perceptrón multicapa con 3 capas desamente conectadas, es decir, con 4 capas ocultas.

*Imagen ejemplo de red densamente conectada, la arquitectura debe tener en la primera capa 3072 neuronas (esto por la dimensión de entrada (32x32x3), las 3 capas adicionales deben tener 512 neuronas y en la final 10 (número de clases)*

In [ ]:
#Las imágenes de 32x32x3 se deben convertir en vectores de 1x3072. Es decir, al final tendremos un vector de
# 50000x3072 (imágenes, pixeles) para entrenamiento y 10000x3072 para prueba.

x_train = np.reshape(x_train, (50000, -1))
x_test = np.reshape(x_test, (10000, -1))

print(f"Dimension de X train:{x_train.shape}")
print(f"Dimension de X test:{x_test.shape}")
print(f"Dimension de y train:{y_train.shape}")
print(f"Dimension de y test:{y_test.shape}")

Ahora es el momento de crear un modelo. Presta mucha atención a la última capa densa y asegúrate de que tenga el número correcto de neuronas para clasificar los diferentes tipos de clases. A continuación, podemos observar la documentación de las diferentes clases que vamos a utilizar:

1. **Dense (Capa densamente conectada):** https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
2. **Input (Capa de entrada):** https://www.tensorflow.org/api_docs/python/tf/keras/Input
3. **Sequential (Modelo secuencial):** https://www.tensorflow.org/api_docs/python/tf/keras/Sequential


In [ ]:
#Se crea la red
num_classes = 10
input_shape = x_train.shape[1]

model_mlp = _________

# Método que imprime el resumen del modelo
model_mlp.summary()

Ahora, debemos seleccionar el **optimizador** del modelo.

Recuerden que el optimizador es un algoritmo utilizado para ajustar los pesos del modelo durante el entrenamiento, con el objetivo de minimizar una función de pérdida o costo. Definen cómo se actualizan los pesos del modelo en función de los gradientes calculados a partir de la función de pérdida.

La selección del optimizador depende el **problema**, no hay uno que funcione mejor que el otro para todas las aplicaciones. El más utilizado es ADAM.

https://www.tensorflow.org/api_docs/python/tf/keras/optimizers
https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam

In [ ]:
optimizer = _________

Ahora, necesitamos compilar el modelo.

El método **compile** en TensorFlow (específicamente en tf.keras, la API de alto nivel para construir y entrenar modelos) configura el modelo para el entrenamiento. Este método especifica tres aspectos fundamentales: el optimizador, la función de pérdida y las métricas a monitorear. Sin compile, un modelo no está listo para el entrenamiento, ya que no tiene definidas las instrucciones sobre cómo optimizar sus pesos para minimizar el error.

Las funciones de pérdida disponibles en Tensorflow se encuentran en:

https://www.tensorflow.org/api_docs/python/tf/keras/losses

In [ ]:
model_mlp.compile(_________)

El método fit en TensorFlow, utilizado con modelos de tf.keras, es el encargado de entrenar el modelo en un conjunto de datos, minimizando la función de pérdida especificada en el paso de compilación (compile).

Principales Argumentos de fit

1. **x (Datos de entrada):** El conjunto de datos de entrada para el entrenamiento. Puede ser un array de NumPy, un tf.data.Dataset, o un generador. Los datos x deben estar en el mismo formato que el modelo espera en su capa de entrada.


2. **y (Etiquetas de salida):** Las etiquetas correspondientes a los datos de entrada x.

3. **batch_size (Tamaño del lote):** Número de muestras que se procesan en una sola pasada antes de actualizar los pesos del modelo. Ajustar el tamaño del lote puede afectar tanto el rendimiento como la memoria usada.

4. **epochs (Épocas):** Número de veces que el modelo verá el conjunto completo de datos de entrenamiento. Cada época implica un recorrido completo por los datos de entrada.

5. **validation_data (Datos de validación):** Se utiliza para evaluar el rendimiento del modelo en un conjunto de datos no visto durante el entrenamiento. Acepta un par (x_val, y_val) o un tf.data.Dataset que contenga los datos de validación.

6. **validation_split:** TensorFlow reserva ese porcentaje de los datos de entrada (x) y las etiquetas (y) para usarlos como datos de validación. Estos datos no se utilizarán para el entrenamiento, sino para evaluar el rendimiento del modelo en cada época. Esta división se realiza antes de cada época y es aleatoria.

In [ ]:
history = _________

In [ ]:
print(history.history.keys())

dict_keys(['accuracy', 'loss', 'precision', 'val_accuracy', 'val_loss', 'val_precision'])


In [ ]:
plot_training_history(_________)

El método [`model.evaluate()`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#evaluate) en Keras se utiliza para evaluar el desempeño de un modelo previamente entrenado en un conjunto de datos de prueba o validación. Este método calcula el valor de la función de pérdida (loss) y las métricas definidas al compilar el modelo, como la precisión (accuracy), en los datos proporcionados.

```
model.evaluate(
    x,               # Conjunto de datos de entrada (puede ser un array, un dataset, o un generador).
    y=None,          # Etiquetas correspondientes (si no están integradas en `x`).
    batch_size=None, # Tamaño del batch. Por defecto, se utiliza el tamaño configurado previamente.
    verbose=1,       # Nivel de información mostrado: 0 (silencioso), 1 (básico), 2 (barras de progreso).
    return_dict=False, # Si es True, devuelve los resultados como un diccionario.
)
```

In [ ]:
scoreFC = ____________________

In [ ]:
plot_10_predictions(____________________)

# **Redes Neuronales Convolucionales**

In [ ]:
from tensorflow.keras.layers import Convolution2D, MaxPooling2D, Activation
from tensorflow.keras.models import Sequential

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

print(f"Dimension de X train:{x_train.shape}")
print(f"Dimension de X test:{x_test.shape}")
print(f"Dimension de y train:{y_train.shape}")
print(f"Dimension de y test:{y_test.shape}")

yname = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

Dimension de X train:(50000, 32, 32, 3)
Dimension de X test:(10000, 32, 32, 3)
Dimension de y train:(50000, 1)
Dimension de y test:(10000, 1)


In [ ]:
# Escalar las imágenes en el rango [0,1]
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

print(f"Dimension de X train:{x_train.shape}")
print(f"Dimension de X test:{x_test.shape}")
print(f"Dimension de y train:{y_train.shape}")
print(f"Dimension de y test:{y_test.shape}")

Dimension de X train:(50000, 32, 32, 3)
Dimension de X test:(10000, 32, 32, 3)
Dimension de y train:(50000, 1)
Dimension de y test:(10000, 1)


### Definir red convolucional

In [ ]:
#Se crea la red y se entrena con train, se valida con test
num_classes = 10

model_cnn = ________________________________________

model_cnn.summary()

In [ ]:
batch_size = 128
epochs = 50

opt = ____________________________________
model_cnn.compile(____________________________________)

history = ____________________________________

In [ ]:
scoreCNN = model_cnn.evaluate(____________________________________)

In [ ]:
plot_training_history(____________________________________)

In [ ]:
plot_10_predictions(____________________________________)

# **Transfer Learning**

El Transfer Learning (o Aprendizaje por Transferencia) es una técnica de aprendizaje automático donde un modelo previamente entrenado en un conjunto de datos o tarea se reutiliza como punto de partida para entrenar un modelo en un nuevo conjunto de datos o una tarea diferente pero relacionada. Es especialmente útil en problemas donde los datos disponibles para entrenar desde cero son limitados.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

print(f"Dimension de X train:{x_train.shape}")
print(f"Dimension de X test:{x_test.shape}")
print(f"Dimension de y train:{y_train.shape}")
print(f"Dimension de y test:{y_test.shape}")

yname = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

Dimension de X train:(50000, 32, 32, 3)
Dimension de X test:(10000, 32, 32, 3)
Dimension de y train:(50000, 1)
Dimension de y test:(10000, 1)


In [ ]:
# Escalar las imágenes en el rango [0,1]
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

print(f"Dimension de X train:{x_train.shape}")
print(f"Dimension de X test:{x_test.shape}")
print(f"Dimension de y train:{y_train.shape}")
print(f"Dimension de y test:{y_test.shape}")

Dimension de X train:(50000, 32, 32, 3)
Dimension de X test:(10000, 32, 32, 3)
Dimension de y train:(50000, 1)
Dimension de y test:(10000, 1)


Realiza los siguientes pasos:

1. Define un objeto llamado base_model que contenga un modelo preentrenado. Para ver los modelos preentrandos que tiene tf.keras sigue el siguiente link: [Modelos Preentrenados](https://www.tensorflow.org/api_docs/python/tf/keras/applications). Escoge uno, y animate a ver la arquitectura de este

2. Como queremos utilizar el modelo ya entrenado, necesitamosconfigurar los pesos del modelo preentrenado estableciendo el parámetro weights como "imagenet".

3. Especifica el tamaño de entrada del modelo con el argumento `input_shape`. **Recuerda** que el tamaño depende de tu conjunto de datos.

4. Asegúrate de excluir las capas superiores del modelo (fully connected layers) para que puedas personalizar la salida según una nueva tarea. Haz esto configurando el parámetro `include_top`



# Argumentos Principales de keras.applications.VGG19

1. **`include_top`** *(bool)*:
   - Indica si se deben incluir las capas fully connected superiores.
   - **Valores**:
     - `True`: Incluye las capas finales del modelo (diseñadas para 1,000 clases de ImageNet).
     - `False`: Excluye las capas superiores (útil para transfer learning).
   - **Predeterminado**: `True`.

2. **`weights`** *(str o None)*:
   - Define qué pesos cargar en el modelo.
   - **Valores**:
     - `"imagenet"`: Usa los pesos preentrenados en ImageNet.
     - `None`: Inicializa pesos aleatorios (entrenamiento desde cero).
   - **Predeterminado**: `"imagenet"`.

3. **`input_tensor`** *(Tensor o None)*:
   - Tensor de entrada opcional para la red (útil si se desea usar un tensor personalizado).
   - **Predeterminado**: `None`.

4. **`input_shape`** *(tuple o None)*:
   - Especifica el tamaño de las entradas en formato `(altura, ancho, canales)`.
   - **Requisitos**:
     - Si `weights="imagenet"`, el tamaño mínimo es `(32, 32, 3)`.
     - Si no se especifica, se usa el predeterminado `(224, 224, 3)`.

5. **`pooling`** *(str o None)*:
   - Tipo de pooling global que se aplica si `include_top=False`.
   - **Valores**:
     - `None`: Sin pooling global.
     - `"avg"`: Usa `GlobalAveragePooling2D`.
     - `"max"`: Usa `GlobalMaxPooling2D`.
   - **Predeterminado**: `None`.

6. **`classes`** *(int)*:
   - Número de clases de salida para la clasificación.
   - **Requisitos**:
     - Solo aplicable si `include_top=True`.
     - Por defecto, es `1000` para las 1,000 categorías de ImageNet.

7. **`classifier_activation`** *(str o None)*:
   - Define la función de activación de la capa de clasificación final.
   - **Valores**:
     - `"softmax"`: Activación estándar para clasificación multiclase.
     - `None`: Sin función de activación (devuelve logits).
   - **Predeterminado**: `"softmax"`.




In [ ]:
base_model = __________________________________

Ahora, como vamos a utilizar los pesos del modelo preentranado. No es necesario entrenar el modelo base, por lo tanto, llama el atributo `trainable` y establecelo como `False`.

In [ ]:
__________________________________

Ahora, vamos a crear un modelo de clasificación usando tu modelo preentrando.

Primero que todo, notarás que en esta ocasión ya no utilizaremos la instrucción `Sequential` de Keras. Por el contrario, vamos a utilizar la clase [`keras.Model`](https://www.tensorflow.org/api_docs/python/tf/keras/Model) para "concatenar" el modelo base y nuestra arquitectura de clasificación.

In [ ]:
# Create inputs with correct shape
inputs = keras.Input(shape=(32,32,3))

x = base_model(inputs, training=False)

# Add pooling layer or flatten layer
x = ____________________________(x)

# Add dense layers
x = ____________________________(x)
x = ____________________________(x)
x = ____________________________(x)
x = ____________________________(x)

# Add output dense last
outputs = ____________________________(x)

# Combine inputs and outputs to create model
model = keras.Model(inputs, outputs)

In [ ]:
model.summary()

In [ ]:
batch_size = 128
epochs = 50
opt     = ____________________________

model.____________________________

history = ____________________________

In [ ]:
scoreCNN = model.evaluate(____________________________________)

In [ ]:
plot_training_history(____________________________________)

In [ ]:
plot_10_predictions(____________________________________)